___
# <center>Regressão logística no scikit-learn, validação cruzada e regularização</center>
___

## Aula 23

**Objetivo.** Ajustar uma regressão logística no scikit-learn, avaliar por
validação cruzada, controlar o sobreajuste com regularização e comparar modelos
sem tocar no teste.

Você vai usar a **base do Projeto 08**, com outra pergunta. O notebook de hoje
continua na aula 24, e os dois juntos são o ponto de partida do projeto.

___
<div id="indice"></div>

## Índice

- [A base](#base)
- [1. Treino, teste e o modelo trivial](#parte1)
- [2. Regressão logística no scikit-learn](#parte2)
- [3. Validação cruzada](#parte3)
- [4. Regularização](#parte4)
- [5. Comparar modelos e abrir a gaveta](#parte5)

___
<div id="base"></div>

# A base

Acórdãos reais de recursos criminais do TJSP, de 2016 a 2023. Cada linha é um
acórdão. A pergunta de hoje: **a decisão teve voto divergente**, ou seja, foi
por maioria?

| coluna | o que é |
|---|---|
| `assunto` | assunto do recurso (roubo majorado, furto qualificado etc.) |
| `camara` | câmara criminal que julgou |
| `comarca` | comarca de origem |
| `polo_mp` | o Ministério Público recorreu (`Ativo`) ou é recorrido (`Passivo`) |
| `ano_publicacao` | ano em que o acórdão foi publicado |
| `tempo` | anos entre a distribuição e o julgamento |
| `rel_idade`, `rel_faculdade`, `rel_tempo_magistratura`, `rel_tipo`, `rel_quinto` | características de quem relatou |
| `negou` | o recurso foi integralmente negado (a resposta do Projeto 08) |
| `unanimidade` | a decisão foi unânime ou por maioria |

In [ ]:
import numpy as np
import pandas as pd
from plotnine import *
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

base = pd.read_csv("https://jtrecenti.github.io/cdad2-202662/ex08-competicao/dados/treino.csv")
base = base.dropna(subset=["unanimidade"])
base["divergiu"] = (base["unanimidade"] == "Maioria").astype(int)
print(base.shape)
base.head(3)

**Por que a coluna `negou` fica de fora.** O resultado do recurso sai no mesmo
julgamento que a divergência: quem quer prever a divergência antes do
julgamento não conhece o resultado. Usar essa coluna seria **vazamento**, tema
da aula 24.

[Volta ao Índice](#indice)

___
<div id="parte1"></div>

# 1. Treino, teste e o modelo trivial

`categoricas` e `numericas` são as colunas que o modelo vai usar. `cria_preparo()`
faz as dummies e a padronização. Rode a célula.

In [ ]:
categoricas = ["assunto", "camara", "comarca", "polo_mp", "rel_faculdade", "rel_tipo", "rel_quinto"]
numericas = ["ano_publicacao", "tempo", "rel_idade", "rel_tempo_magistratura"]

def cria_preparo():
    """Dummies nas categóricas e padronização nas numéricas, com faltantes preenchidos."""
    return make_column_transformer(
        (make_pipeline(SimpleImputer(strategy="most_frequent"),
                       OneHotEncoder(handle_unknown="ignore", min_frequency=5)), categoricas),
        (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numericas),
    )

O teste vai para a **gaveta**: separamos 25% dos acórdãos e não olhamos para eles
até o fim. Complete a proporção do teste. `stratify` mantém a mesma taxa de
divergência nas duas partes.

In [ ]:
colunas = categoricas + numericas
X_treino, X_teste, y_treino, y_teste = train_test_split(
    base[colunas], base["divergiu"], test_size=____, stratify=base["divergiu"], random_state=23)
print(len(X_treino), len(X_teste), y_treino.mean().round(3))

**Pergunta.** Qual seria a acurácia do modelo trivial, que diz que nunca há
divergência? Por que a acurácia não serve para este problema?

[Volta ao Índice](#indice)

___
<div id="parte2"></div>

# 2. Regressão logística no scikit-learn

`make_pipeline` junta o preparo e o modelo num objeto só. Com `C=10000`, a
penalidade é quase nula, e o modelo fica perto do que o statsmodels daria.
`fit` ajusta, e `predict_proba` dá as probabilidades: a coluna `[:, 1]` é a
probabilidade de divergência. Complete o nome do método que ajusta o modelo.

In [ ]:
quase_sem_penalidade = make_pipeline(cria_preparo(), LogisticRegression(C=10000, max_iter=3000))
quase_sem_penalidade.____(X_treino, y_treino)

prob_treino = quase_sem_penalidade.predict_proba(X_treino)[:, 1]
print("AUC no treino:", round(roc_auc_score(y_treino, prob_treino), 3))

**Pergunta.** Uma AUC de 0,9 no treino é uma boa notícia? Pense no exercício 4 da
aula 22, antes de olhar a próxima parte.

[Volta ao Índice](#indice)

___
<div id="parte3"></div>

# 3. Validação cruzada

A validação cruzada estima o desempenho em casos novos **usando só o treino**.
Complete o número de partes (5) e a medida (`"roc_auc"`).

In [ ]:
pastas = StratifiedKFold(____, shuffle=True, random_state=23)
auc = cross_val_score(quase_sem_penalidade, X_treino, y_treino, cv=pastas, scoring="____")
print(auc.round(3))
print("média:", auc.mean().round(3))

**Pergunta.** Compare a AUC no treino da parte 2 com a média da validação
cruzada. O que a diferença indica?

[Volta ao Índice](#indice)

___
<div id="parte4"></div>

# 4. Regularização

No scikit-learn, a penalidade é controlada por `C`: **`C` menor, penalidade
maior**. O laço abaixo calcula a AUC por validação cruzada para vários valores
de `C`. Complete o valor que vai para o modelo.

In [ ]:
linhas = []
for C in [0.003, 0.01, 0.03, 0.1, 0.3, 1, 10, 100, 10000]:
    modelo = make_pipeline(cria_preparo(), LogisticRegression(C=____, max_iter=3000))
    auc = cross_val_score(modelo, X_treino, y_treino, cv=pastas, scoring="roc_auc").mean()
    linhas.append({"C": C, "auc_validacao": round(auc, 3)})
por_c = pd.DataFrame(linhas)
por_c

In [ ]:
(ggplot(por_c, aes(x="C", y="auc_validacao"))
 + geom_line(color="#E4002B", size=1) + geom_point(color="#E4002B", size=3)
 + scale_x_log10()
 + labs(x="C (mais penalidade para a esquerda)", y="AUC na validação cruzada")
 + theme_minimal())

**Perguntas.**

1. Qual `C` teve a maior AUC na validação cruzada?
2. O que acontece com a AUC quando a penalidade é muito fraca? E quando é forte
   demais? Use as palavras **sobreajuste** e **subajuste**.

[Volta ao Índice](#indice)

___
<div id="parte5"></div>

# 5. Comparar modelos e abrir a gaveta

Variáveis também se escolhem por validação cruzada. Comparamos dois modelos com
a mesma penalidade: um com todas as colunas e um sem `comarca`, que tem mais de
300 categorias.

In [ ]:
sem_comarca = [c for c in colunas if c != "comarca"]
preparo_sem = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="most_frequent"),
                   OneHotEncoder(handle_unknown="ignore", min_frequency=5)),
     [c for c in categoricas if c != "comarca"]),
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numericas),
)
com = make_pipeline(cria_preparo(), LogisticRegression(C=0.3, max_iter=3000))
sem = make_pipeline(preparo_sem, LogisticRegression(C=0.3, max_iter=3000))
print("com comarca:", cross_val_score(com, X_treino, y_treino, cv=pastas, scoring="roc_auc").mean().round(3))
print("sem comarca:", cross_val_score(sem, X_treino, y_treino, cv=pastas, scoring="roc_auc").mean().round(3))

Agora, e só agora, abrimos a gaveta. Ajuste o modelo final no treino inteiro e
calcule a AUC no teste. Complete com a base certa.

In [ ]:
final = make_pipeline(cria_preparo(), LogisticRegression(C=0.3, max_iter=3000))
final.fit(X_treino, y_treino)
prob_teste = final.predict_proba(____)[:, 1]
print("AUC no teste:", round(roc_auc_score(y_teste, prob_teste), 3))

**Perguntas.**

1. A AUC no teste ficou perto da validação cruzada? O que isso diz sobre a
   estimativa da validação?
2. Por que não seria correto testar vários `C` diretamente no teste e relatar o
   melhor número?

[Volta ao Índice](#indice)